# 04 — Morphology, clinical reading, Napari

A Dice of 0.80 can hide two very different masks: one that misses capillaries (low sensitivity) and one that paints choroidal noise (low precision). Skeleton metrics separate those cases.

| Quantity | Construction | Clinical reading |
| --- | --- | --- |
| **Density** | vessel pixels / FOV pixels | rarefaction vs hyperperfusion |
| **Length** | 8-connected centreline (diag = √2) | drops if branches are broken |
| **Width** | 2 × distance transform on the skeleton | arteriolar narrowing |
| **Branch points** | skeleton pixels with ≥3 neighbours | extra twigs from false positives |
| **Tortuosity** | arc length / chord per segment | hypertension, plus some DR phenotypes |

This notebook expects `pixi run demo` (or at least classical + evaluate) so predictions exist. If they are missing it falls back to Frangi on the fly.


In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().parents[0] / "src") if Path.cwd().name == "notebooks" else "src")

import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from skimage.morphology import skeletonize

from drive_seg.classical import segment_frangi
from drive_seg.data import iter_samples, load_sample, resolve_dataset_root
from drive_seg.io_pred import load_prediction
from drive_seg.morphology import morphology_metrics
from drive_seg.paths import SYNTHETIC_DIR
from drive_seg.synthetic import generate_synthetic_dataset
from drive_seg.viz import overlay_rgb

if not (SYNTHETIC_DIR / "training" / "images").is_dir():
    generate_synthetic_dataset()
root = resolve_dataset_root()
sample = iter_samples(root, split="test")[0]
image, vessel, fov = load_sample(sample)

def mask_for(method):
    try:
        return load_prediction(method, sample.sample_id, sample.split)
    except FileNotFoundError:
        if method == "frangi":
            pred, _ = segment_frangi(image, fov)
            return pred
        return None

methods = {name: mask_for(name) for name in ("frangi", "rf", "svm", "unet")}
methods = {k: v for k, v in methods.items() if v is not None}
print("available methods", list(methods))

## Skeleton overlay

The skeleton is the object morphology actually uses. Broken centreline → underestimated length and extra end-points. Spurious loops → extra branch points.

In [ ]:
skel = skeletonize(vessel)
fig = make_subplots(rows=1, cols=2, subplot_titles=("ground-truth vessels", "skeleton"))
fig.add_trace(go.Heatmap(z=vessel.astype(float), colorscale="Gray", showscale=False), 1, 1)
fig.add_trace(go.Heatmap(z=skel.astype(float), colorscale="Gray", showscale=False), 1, 2)
fig.update_xaxes(visible=False)
fig.update_yaxes(visible=False, autorange="reversed")
fig.update_layout(template="plotly_white", height=360, title=sample.sample_id)
fig.show()

In [ ]:
rows = [dict(method="ground_truth", **morphology_metrics(vessel, fov))]
for name, pred in methods.items():
    rows.append(dict(method=name, **morphology_metrics(pred, fov)))
morph = pd.DataFrame(rows)
print(morph[["method", "vessel_density", "length_px", "mean_width_px", "n_branch_points", "mean_tortuosity"]].round(3))

long = morph.melt(id_vars=["method"],
                  value_vars=["vessel_density", "length_px", "mean_width_px", "n_branch_points", "mean_tortuosity"],
                  var_name="quantity", value_name="value")
fig = px.bar(long, x="method", y="value", facet_col="quantity", facet_col_wrap=3,
             title="Morphology vs ground truth (one test image)")
fig.update_layout(template="plotly_white")
fig.update_yaxes(matches=None)
fig.show()

## How to read a disagreement

- **High Dice, too many branch points** — salt-and-pepper false positives. Clinically you would not trust a branching index.
- **High precision, low sensitivity** — Frangi's usual failure: thick trunks only. A width statistic will be biased high because capillaries never entered the mask.
- **Tortuosity ≫ ground truth** — wiggly false traces, or a broken skeleton that turned one vessel into many short, bent fragments.

These caveats still apply on official DRIVE. Quote morphology only after looking at the overlay.


## RGB overlay (Plotly)

Green TP, red FP, blue FN — same convention as `results/figures/overlay_*.png`.

In [ ]:
pred = methods.get("unet") or methods.get("rf") or methods.get("frangi")
ov = overlay_rgb(image, pred, vessel, fov)
fig = make_subplots(rows=1, cols=2, subplot_titles=("fundus", "overlay"))
fig.add_trace(go.Image(z=image), 1, 1)
fig.add_trace(go.Image(z=ov), 1, 2)
fig.update_xaxes(visible=False)
fig.update_yaxes(visible=False)
fig.update_layout(template="plotly_white", height=380, title="U-Net (or best available) overlay")
fig.show()

## Napari (interactive)

Run this cell on a **desktop** (Windows, or Linux with a display). It adds the fundus as an RGB image and each mask as a labels layer. Toggle layers in the left dock.

Headless agents should skip it or call `scripts/visualize_napari.py --headless`.


In [ ]:
# Interactive viewer — skip on CI / SSH without DISPLAY.
import os, sys
has_gui = sys.platform == "win32" or bool(os.environ.get("DISPLAY") or os.environ.get("WAYLAND_DISPLAY"))
if not has_gui:
    print("No GUI display; use `pixi run napari` on a workstation.")
else:
    import napari
    viewer = napari.Viewer(title=f"DRIVE vessels — {sample.sample_id}")
    viewer.add_image(image, name="fundus", rgb=True)
    viewer.add_labels(vessel.astype(int), name="ground_truth")
    for name, pred in methods.items():
        viewer.add_labels(pred.astype(int), name=name)
    print("Close the Napari window to continue.")
    napari.run()